In [1]:
import duckdb
import geopandas as gpd
import pystac
from create_buffered_tile import (
    get_stac_item,                                     
    get_buffered_bbox,
    get_distance_degrees
)

In [2]:
item_id = 'N075E299_LAS_Phase2.copc'
collection = 'laz-phase2'
stac = 'https://spved5ihrl.execute-api.us-west-2.amazonaws.com'
item = get_stac_item(item_id, collection, stac) 
# print(f'item: {item.bbox}')


In [3]:
buffer = 30
distance = get_distance_degrees(buffer)
bbox_buffer = get_buffered_bbox(item.bbox, distance)
print(f'bbox: {bbox_buffer}')

bbox: [-84.53558049335248, 38.2605455066475, -84.51739850664751, 38.2749934933525]


In [4]:
# Define constants for our read
OVERTURE_RELEASE = "2026-07-22.0"
SOURCE_DATA_URL = f"s3://overturemaps-us-west-2/release/{OVERTURE_RELEASE}/theme=buildings/type=building/*"
OUTPUT_FILE = "/home/ian/repos/lidar-classify-by-features/buildings/N075E299.parquet"

xmax = bbox_buffer[2]
xmin = bbox_buffer[0]
ymax = bbox_buffer[3]
ymin = bbox_buffer[1]


# Read via DuckDB instead of Spark

No JVM needed. `httpfs` gives DuckDB S3 access (Overture's bucket is public, so no credentials needed — just the region), `spatial` gives it geometry functions if we need them later. Filtering on `bbox.xmin/xmax/ymin/ymax` prunes files/row-groups the same way it did in the Spark version, without touching the actual geometry column.

`geometry` comes back from Overture's parquet as raw WKB (`bytearray`) — that's their on-disk format, not a DuckDB-native type, so no spatial function calls are required just to read it. `.apply(bytes)` before `gpd.GeoSeries.from_wkb(...)` is needed because DuckDB's Python API hands back `bytearray`, and `from_wkb` wants real `bytes`.

In [5]:
print(SOURCE_DATA_URL)

con = duckdb.connect()
con.execute("INSTALL spatial;")
con.execute("INSTALL httpfs;")
con.execute("LOAD spatial;")
con.execute("LOAD httpfs;")
con.execute("SET s3_region='us-west-2';")

query = f"""
-- COPY (
SELECT id, height, geometry
FROM read_parquet('{SOURCE_DATA_URL}', filename=true, hive_partitioning=1)
WHERE bbox.xmin <= {xmax}
  AND bbox.xmax >= {xmin}
  AND bbox.ymin <= {ymax}
  AND bbox.ymax >= {ymin}
--  TO '{OUTPUT_FILE}' (FORMAT PARQUET)
"""

buildings_df = con.execute(query).df()
print(f"{len(buildings_df):,} building footprints found")
buildings_df.head()

s3://overturemaps-us-west-2/release/2026-07-22.0/theme=buildings/type=building/*


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

73 building footprints found


,id,height,geometry
0,d5afe478-0f3a-4078-8d7b-02d752e4ea20,5.473471,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 9, 0, 0, 0, 119, 7..."
1,0b43c56f-423b-481b-b104-2306b6d8d71c,NaN,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 5, 0, 0, 0, 72, 90..."
2,2681a768-4ef0-447b-a771-197fab607fd5,3.071295,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 7, 0, 0, 0, 82, 22..."
3,65dae9e5-3d21-440d-8df9-e92ae46b1ff0,NaN,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 7, 0, 0, 0, 124, 1..."
4,b02cdccb-53cc-4949-bf57-5f62edbeaf87,NaN,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 5, 0, 0, 0, 204, 2..."


In [6]:
buildings_gdf = gpd.GeoDataFrame(
    buildings_df[["id", "height"]],
    geometry=gpd.GeoSeries.from_wkb(buildings_df["geometry"].apply(bytes)),
    crs="EPSG:4326",
)

print(buildings_gdf.geom_type.value_counts())
print("bounds:", buildings_gdf.total_bounds)

buildings_gdf.to_parquet(OUTPUT_FILE)
print(f"wrote {len(buildings_gdf):,} rows to {OUTPUT_FILE}")

buildings_gdf.head()

Polygon    73
Name: count, dtype: int64
bounds: [-84.5421091  38.2562889 -84.5167263  38.2752007]
wrote 73 rows to /home/ian/repos/lidar-classify-by-features/buildings/N075E299.parquet


,id,height,geometry
0,d5afe478-0f3a-4078-8d7b-02d752e4ea20,5.473471,"POLYGON ((-84.52629 38.27051, -84.52601 38.270..."
1,0b43c56f-423b-481b-b104-2306b6d8d71c,NaN,"POLYGON ((-84.52618 38.26649, -84.52617 38.266..."
2,2681a768-4ef0-447b-a771-197fab607fd5,3.071295,"POLYGON ((-84.5272 38.2696, -84.5273 38.26946,..."
3,65dae9e5-3d21-440d-8df9-e92ae46b1ff0,NaN,"POLYGON ((-84.52716 38.26671, -84.52722 38.266..."
4,b02cdccb-53cc-4949-bf57-5f62edbeaf87,NaN,"POLYGON ((-84.52828 38.26667, -84.52828 38.266..."
